# Optuna tuning — 10 deep forecasters on Google Colab

`seq_len = 96`, `pred_len = 1`, **50 trials per model**, **3 repeats per trial**
during the search, final run with **`--itr 5`**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Run the cells in order. Steps 2–4 are setup; step 6 is the search and is the
long one. Everything is written to Google Drive, and studies **resume**: if
Colab disconnects, re-run the setup cells and then step 6 again. A model
stopped at 30 of its 50 trials runs the remaining 20, and a model that already
finished is skipped — re-running the cell costs nothing for work already
done.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — switch Runtime > Change runtime type > T4 GPU (the search will be very slow on CPU)'

## 2. Mount Drive

Results and the Optuna database go here so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ProjectC_tuning'   # <- change if you like
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('results ->', DRIVE_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is public, so this needs no credentials. Re-running the cell
in a later session updates an existing clone instead of failing.

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/optuna-hyperparameter-tuning-11-models-hc0tgt'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn and matplotlib. These
are the extras this repo needs — `fast_pytorch_kmeans` for AdaWaveNet,
`reformer-pytorch`/`local-attention`
because `layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q optuna einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch, optuna
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| optuna', optuna.__version__)

## 5. Sanity check (~1 minute)

Two trials of the cheapest model. If this prints a best validation loss, the
environment is wired up correctly.

In [ ]:
!python tuning/optuna_tune.py --model DLinear --n_trials 2 --train_epochs 3 \
    --out_dir /content/_smoketest --checkpoint_dir /content/_ckpt 2>&1 | tail -5

## 6. The search — 50 trials per model, 3 repeats each

One model per call, so progress is saved after each. `N_TRIALS` is a **target
per model**, not a batch size: the driver counts what the database already
holds and runs only the remainder, so this cell is safe to re-run as often as
you like. Editing `MODELS` lets you run a subset — e.g. the cheap ones first.

Every configuration is trained **3 times** (seeds 2021-2023) and scored by the
mean of its validation losses — `run.py --itr` applied during the search. One
run on a 519-window validation split is too noisy to rank on: the seed spread
is about the same size as the gap between neighbouring configurations.

So a model's search is up to 50 x 3 = 150 trainings, though fewer in practice
because a trial pruned on its first seed never pays for the other two. Rough
cost on a T4: DLinear/FITS ~10-20 minutes each, PatchTST/TSLANet/iTransformer/
ModernTCN/AdaWaveNet/TimeMixer an hour or more, MSGNet/TimesNet several
hours. Plan on running this across more than one session — it resumes.

Checkpoints go to local disk (`/content/_ckpt`), not Drive — they are
rewritten every improving epoch and deleted after each trial, so putting them
on a network mount would dominate the runtime.

In [ ]:
MODELS = ['DLinear', 'FITS', 'TSLANet', 'PatchTST', 'iTransformer', 'ModernTCN',
          'AdaWaveNet', 'TimeMixer', 'MSGNet', 'TimesNet']
N_TRIALS = 50
N_SEEDS  = 3     # repeats per trial; the objective is their mean

import subprocess, time
for model in MODELS:
    print(f'\n{"="*72}\n  {model}\n{"="*72}', flush=True)
    started = time.time()
    subprocess.run(['python', 'tuning/optuna_tune.py',
                    '--model', model,
                    '--n_trials', str(N_TRIALS),
                    '--n_seeds', str(N_SEEDS),
                    '--out_dir', DRIVE_DIR,
                    '--checkpoint_dir', '/content/_ckpt'])
    print(f'{model} finished in {(time.time()-started)/60:.1f} min', flush=True)

### Optional: a fast indicative pass

If you just want to see the landscape quickly, `--n_seeds 1` trains each
configuration once and searches about three times faster. The ranking is then
within seed noise, so use it to explore, not to report. Give it its own
`--out_dir` so it does not mix with the 3-seed studies, whose objective is on a
different footing.

In [ ]:
# for model in MODELS:
#     subprocess.run(['python', 'tuning/optuna_tune.py', '--model', model,
#                     '--n_trials', str(N_TRIALS), '--n_seeds', '1',
#                     '--out_dir', DRIVE_DIR + '_fast',
#                     '--checkpoint_dir', '/content/_ckpt'])

## 7. Final runs — best config, `--itr 5`

Re-trains each winner through `run.py` with **five seeds (2021–2025)** and
prints mean ± std, min and max for every HAR-comparable metric — MSE/MAE on
the `ln(RV)` scale, QLIKE and MSE_RV/MAE_RV back on the variance scale, all
directly comparable with `python HAR-RV_RUN.PY --log`.

In [ ]:
import json, glob, subprocess, os

for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    best = json.load(open(path))
    argv = best['command'].split()[3:] + ['--des', 'best', '--itr', '5']
    print(f'\n{"="*72}\n  {best["model"]}  (best val loss {best["best_val_loss"]:.6f})\n{"="*72}', flush=True)
    subprocess.run(['python', '-u', 'run.py'] + argv)

## 8. Summary table

Best configuration per model, ranked by validation loss.

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    b = json.load(open(path))
    rows.append({'model': b['model'],
                 'val_loss': round(b['best_val_loss'], 6),
                 'val_std': b.get('best_val_loss_std'),   # spread over the trial's seeds
                 'trials': b['n_complete'],
                 'minutes': round(b['seconds'] / 60, 1),
                 **b['best_params']})

df = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
df.to_csv(os.path.join(DRIVE_DIR, 'best_configs.csv'), index=False)
display(df[['model', 'val_loss', 'val_std', 'trials', 'minutes']])
df

In [ ]:
# the exact command line that reproduces each winner
for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    b = json.load(open(path))
    print(f'# {b["model"]}  (val {b["best_val_loss"]:.6f})\n{b["command"]}\n')

## Notes

* **Resuming** — studies live in `optuna.db` on Drive. Re-running step 6 after
  a disconnect tops each study up to `N_TRIALS` and skips those already there.
  Raise `N_TRIALS` to search further; the extra trials build on the existing
  history rather than starting a new study. A trial interrupted mid-training
  is the only thing lost, and it is reported on the next run.
* **What is being minimised** — the lowest validation loss reached during
  training, i.e. the epoch `EarlyStopping` checkpoints. The test split is
  never touched during a search; it appears only in step 7.
* **Equal protocol** — every model gets the same 50 trials, the same 3
  repeats per trial and the same learning-rate/batch/schedule ranges, so the
  table compares architectures, not tuning effort.
* **Two kinds of repeat** — `--n_seeds 3` during the search (validation,
  averaged, picks the winner) and `--itr 5` for the final run (test, reported
  as mean +/- std). Only the first ranks anything.
* **Search spaces** — `tuning/README.md` documents every range and every
  architectural constraint; `tuning/search_spaces.py` is the source of truth.